# 07 — Styling: Colours and Line Widths

Every renderer in eucare reads three attribute keys off graph elements:

- `obj['color_key']` — a colour. Either an `(r, g, b)` / `(r, g, b, a)` tuple in `[0, 1]`, a hex string, or any hashable value (gets hashed into a deterministic random colour).
- `edge['line_width']` and `vertex['line_width']` — per-element stroke widths in figure units.
- `face['color_key']` — face fill colour (only applied when `render_faces=True`).

This notebook shows three ways to set those keys:

1. *Automatically* via classifiers in [`eucare.classifiers`](../reference/eucare/classifiers.md).
2. *By hand* on faces and edges chosen by predicate.
3. *Inherited* via the `pre_conway` attribute that Conway operators leave behind on every new vertex / face / edge.

In [ ]:
import os
os.environ.setdefault('TQDM_DISABLE', '1')

import matplotlib
matplotlib.rcParams['figure.figsize'] = (5, 5)
import matplotlib.pyplot as plt
import numpy as np

import eucare as ec
from eucare import (
    classifiers,
    colorization,
    conway,
    example_graphs,
    example_tilesets,
    overlap,
    reciprocal_figures,
    rendering,
)
from eucare.rendering import multi_show


## Defaults: the crease-pattern preset

`rendering.CREASE_PATTERN_PRESET` is the dict we've been using throughout the series; alongside it sit three colour constants matching common origami conventions.

In [ ]:
print(rendering.CREASE_PATTERN_PRESET)
print('mountain:', rendering.MOUNTAIN_COLOR,
      'valley:', rendering.VALLEY_COLOR,
      'flat:', rendering.FLAT_COLOR)


## Automatic colouring 1: by number of corners

We use `t_4_6_12` — squares (4), hexagons (6), dodecagons (12) — as the testbed. `LenClassifier` plus a one-line `lambda` extracts the corner count and `colorize` writes the result into `face['color_key']`. The renderer picks any hashable as a deterministic colour seed, so each corner count maps to its own stable colour.

In [ ]:
from eucare.classifiers import LenClassifier, PreMapClassifier

G = example_graphs.from_tiles(example_tilesets.t_4_6_12(), rings=2)
G.recompute_lengths_and_angles()
G_corners = G.copy()
corner_classifier = PreMapClassifier(LenClassifier(),
                                     lambda f: list(f.halfedge_iter()))
colorization.colorize(G_corners, corner_classifier)
multi_show([G, G_corners],
           titles=['plain', 'colored by corner count'],
           render_faces=True, face_inset=0.05, render_vertices=False)


## Automatic colouring 2: by congruence

In [ ]:
G = example_graphs.from_tiles(example_tilesets.t_4_6_12(), rings=2)
G.recompute_lengths_and_angles()
hexagons_and_dodecagons = [f for f in G.faces if f.order() == 6 or f.order() == 12]
G = conway.kis_graph()(G, faces=hexagons_and_dodecagons, delete_on_border=False)
G.recompute_lengths_and_angles()

G_len  = G.copy(); colorization.colorize(G_len, corner_classifier)
G_cong = G.copy(); colorization.congruency_colorize(G_cong)

n_len  = len({f['color_key'] for f in G_len.faces  if 'color_key' in f.attributes})
n_cong = len({f['color_key'] for f in G_cong.faces if 'color_key' in f.attributes})
print(f'classes — by len: {n_len}, by congruence: {n_cong}')

multi_show([G_len, G_cong],
           titles=[f'by corner count ({n_len} classes)',
                   f'by congruence ({n_cong} classes)'],
           render_faces=True, face_inset=0.05, render_edges=False, render_vertices=False)


## Hand-assigned face colours

For one-off highlights, just write to `face['color_key']` directly. Pick faces with any predicate; below we paint everything within a small radius of the origin orange.

In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(6), rings=2)
G.recompute_lengths_and_angles()
for f in G.faces:
    if np.linalg.norm(f.midpoint()) < 1.5:
        f['color_key'] = (1.0, 0.6, 0.0, 0.9)  # opaque orange
G.show(render_faces=True, face_inset=0.05, render_vertices=False)


## Hand-assigned edge colours and widths

Per-edge `'color_key'` and `'line_width'` work the same way. Below we make the boundary halfedges of one chosen face thick and red. Note that we set both directions of every edge.

In [ ]:
G = example_graphs.from_tiles(example_tilesets.platonic(4), rings=3)
G.recompute_lengths_and_angles()
central = G.central_face()
for h in central.halfedge_iter():
    h['color_key'] = h.rev['color_key'] = (0.85, 0.10, 0.10, 1.0)
    h['line_width'] = h.rev['line_width'] = 0.2
G.show(face_inset=0.05, render_vertices=False)


## Inheriting colours via `pre_conway`

Every Conway operator stores some back-pointers on its outputs: some of the new vertex / face / halfedges in the result graph carry `obj['pre_conway']` → the corresponding element of the *input* graph. We can use that to propagate any per-face attribute through an operator, for example for coloring

In [ ]:
from eucare.half import Face

G = example_graphs.from_tiles(example_tilesets.platonic(4), rings=1)
central_face = G.central_face()
central_face['color_key'] = (1.0, 0.5, 0.1, 0.9)

for v in central_face.vertex_iter():
    v['color_key'] = np.random.rand(3)

D = conway.chamfer_graph()(G.copy(), delete_on_border=False)
D.recompute_lengths_and_angles()
for obj in D.vertices.union(D.faces).union(D.halfedges):
    src = obj.attributes.get('pre_conway')
    if isinstance(src, Face) and 'color_key' in src.attributes:
        obj['color_key'] = src['color_key']

multi_show([G, D],
           titles=['original', 'dual (colors inherited from original)'],
           render_faces=True, render_vertices=True, face_inset=0,
           vertex_radius=0.1)


## What's next

- [`08_Modifications`](08_Modifications.ipynb) — surgery on the graph (delete / subdivide / coordinate transforms).
- The legacy [`Tiling Demo`](Tiling%20Demo.ipynb) notebook shows more elaborate styling tied to specific tilings.